# Luganda AI — QLoRA fine-tune of Sunflower-14B (Kaggle, private)

Same training as on RunPod, adapted for Kaggle's free NVIDIA GPUs.

**Before you run, set these (top-right / Notebook settings):**
1. **Accelerator:** `GPU T4 x2` (two T4s give 32 GB total — needed for a 14B model).
2. **Internet:** ON (to download the model from Hugging Face).
3. **Add-ons -> Secrets:** add a secret named `HF_TOKEN` = your Hugging Face read token.
4. **Add-ons -> Upload / Datasets:** create a **private** dataset named `lugandaai-v03`
   containing `train-v0.3.jsonl` and `pilot-p1-0.7-draft.jsonl`, and attach it.
   Its files then appear under `/kaggle/input/lugandaai-v03/`.

T4s do **not** support bf16, so this notebook detects that and uses fp16 automatically.
The trained adapter is saved to `/kaggle/working/` — download it when done.


In [ ]:
# 1) Install the libraries (Kaggle's base image may be older)
!pip -q install -U "transformers>=4.51" accelerate peft bitsandbytes datasets sentencepiece protobuf
print("installed")

In [ ]:
# 2) Hugging Face login from the Kaggle secret HF_TOKEN
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=token)
print("Logged in as:", whoami()["name"])

In [ ]:
# 3) Config, data loading, and safety checks
import json, re, os

BASE_MODEL = "Sunbird/Sunflower-14B"
DATA_PATH  = "/kaggle/input/lugandaai-v03/train-v0.3.jsonl"
BENCH_PATH = "/kaggle/input/lugandaai-v03/pilot-p1-0.7-draft.jsonl"
OUT_DIR    = "/kaggle/working/sunflower-v0.3"
MAX_SEQ_LEN = 768          # drop to 512 if you hit out-of-memory
EPOCHS = 1
RUN_BENCHMARK = True       # after training, answer the 29 frozen questions

# held-out clock hours must never appear in training (D-020)
HELDOUT = re.compile(r"\b(8:00am|9:00am|10:00am|1:00pm|2:00pm)\b")
CREDS = [re.compile(p) for p in (r"\bhf_[A-Za-z0-9]{20,}", r"\bsk-[A-Za-z0-9]{20,}", r"SUNBIRD_API_TOKEN", r"\b[A-Fa-f0-9]{40,}\b")]

def load_rows(path):
    rows=[]
    with open(path, encoding="utf-8") as fh:
        for i, line in enumerate(fh, 1):
            line=line.strip()
            if not line: continue
            r=json.loads(line)
            msgs=r.get("messages")
            assert isinstance(msgs,list) and len(msgs)>=2, f"line {i}: bad messages"
            assert msgs[0]["role"]=="user" and msgs[-1]["role"]=="assistant", f"line {i}: roles"
            blob=" ".join(str(m.get("content","")) for m in msgs)
            assert not HELDOUT.search(blob), f"line {i}: held-out hour leaked"
            assert not any(p.search(blob) for p in CREDS), f"line {i}: credential-like string"
            rows.append(r)
    assert rows, "no rows"
    return rows

rows = load_rows(DATA_PATH)
from collections import Counter
print("rows:", len(rows), "| by source:", dict(Counter(r.get("source","?") for r in rows)))

In [ ]:
# 4) Load Sunflower-14B in 4-bit, attach LoRA, and train
import torch
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          TrainingArguments, Trainer, DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
import dataclasses

# T4 has no bf16 -> use fp16. Newer cards (A100/4090) -> bf16.
use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("GPU:", torch.cuda.get_device_name(0), "| bf16 supported:", use_bf16, "| using", compute_dtype)

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def encode(r):
    text = tok.apply_chat_template(r["messages"], tokenize=False, add_generation_prompt=False)
    enc = tok(text, truncation=True, max_length=MAX_SEQ_LEN)
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"]}

dataset = Dataset.from_list([encode(r) for r in rows])
print("prepared", len(dataset), "examples")

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=compute_dtype)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=bnb,
                                             torch_dtype=compute_dtype, device_map="auto")
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)
lora = LoraConfig(task_type="CAUSAL_LM", r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
                  target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])
model = get_peft_model(model, lora)
model.print_trainable_parameters()

desired = dict(output_dir=OUT_DIR, num_train_epochs=EPOCHS, per_device_train_batch_size=1,
               gradient_accumulation_steps=16, learning_rate=2e-4, warmup_ratio=0.03,
               weight_decay=0.0, lr_scheduler_type="cosine", logging_steps=10,
               save_strategy="epoch", seed=7, optim="paged_adamw_8bit", report_to=[],
               bf16=use_bf16, fp16=not use_bf16)
allowed = {f.name for f in dataclasses.fields(TrainingArguments)}
kept = {k:v for k,v in desired.items() if k in allowed}
targs = TrainingArguments(**kept)

collator = DataCollatorForLanguageModeling(tokenizer=tok, mlm=False)
trainer = Trainer(model=model, args=targs, train_dataset=dataset, data_collator=collator)
trainer.train()
model.save_pretrained(OUT_DIR)
tok.save_pretrained(OUT_DIR)
print("Training complete. Adapter saved to", OUT_DIR)

In [ ]:
# 5) (optional) Answer the 29 frozen benchmark questions with the fine-tuned model
if RUN_BENCHMARK and os.path.exists(BENCH_PATH):
    import json
    items=[]
    for line in open(BENCH_PATH, encoding="utf-8"):
        line=line.strip()
        if not line: continue
        it=json.loads(line)
        if it.get("review_status")=="excluded" or it.get("turns"): continue
        if (it.get("prompt") or "").strip():
            items.append(it)

    def enc_gen(msgs):
        r=None
        for kw in (dict(enable_thinking=False, return_dict=True), dict(return_dict=True), dict()):
            try:
                r=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", **kw); break
            except TypeError: continue
        return {k:v.to(model.device) for k,v in r.items()} if hasattr(r,"keys") else {"input_ids": r.to(model.device)}

    model.eval()
    out="/kaggle/working/answers-v0.3.jsonl"
    with open(out,"w",encoding="utf-8") as fh:
        for i,it in enumerate(items,1):
            e=enc_gen([{"role":"user","content":it["prompt"]}]); plen=e["input_ids"].shape[1]
            with torch.no_grad():
                g=model.generate(**e, max_new_tokens=400, do_sample=False, pad_token_id=tok.eos_token_id)
            ans=tok.decode(g[0][plen:], skip_special_tokens=True).strip()
            fh.write(json.dumps({"item_id":it.get("id"),"category":it.get("category",""),
                                 "prompt":it["prompt"],"expected_behavior":it.get("expected_behavior",""),
                                 "model":"v0.3","answer":ans}, ensure_ascii=False)+"\n")
            print(f"[{i}/{len(items)}] {it.get('id')}")
    print("Wrote", out)
else:
    print("Benchmark skipped (set RUN_BENCHMARK=True and attach the dataset).")

## Download your results

In the right panel under **Output** (`/kaggle/working/`):
- `sunflower-v0.3/` — the trained adapter (download the folder).
- `answers-v0.3.jsonl` — the model's answers to the 29 questions (for the blind comparison).

Nothing here is public: a private notebook + private dataset keep your data and model yours.
